# Phase 1 preflight: causal Mimi audio tokens

This notebook validates the audio side of the fixed-schedule Step 1 planner **before planner implementation**. It does not modify checkpoints or precompute the dataset.

The required gates are:

1. local Moshi source and Mimi checkpoint are loadable without a Hugging Face download;
2. the checkpoint identity, 24 kHz / 12.5 Hz timing, 8 codebooks, and 2,048-entry vocabulary match our contract;
3. real SuSuInterActs audio/motion pairs have valid durations and sample rates;
4. offline and causal-streaming Mimi q0 tokens agree;
5. changing future audio cannot change past q0 tokens;
6. fixed `[gap_3]` anchors align exactly: 4 body-token frames at 10 Hz = 5 Mimi frames at 12.5 Hz = 0.4 s; and
7. the existing Qwen checkpoint is internally consistent and contains the 8,192 planned body-token IDs.

Run all cells in order. The final table is the go/no-go report. Listening to the optional q0 reconstruction is a qualitative sanity check, not proof that q0 alone is sufficient for motion planning.


## 1. Configuration

Defaults match the current local layout. On the remote server, set `MOSHI_REPO`, `MIMI_WEIGHT`, or `SUSU_ROOT` as environment variables instead of editing code. Use `MIMI_DEVICE=cuda` when a GPU is available.


In [ ]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import math
import os
import platform
import sys
import time
import wave
from collections import Counter
from fractions import Fraction
from pathlib import Path


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'motion_generation').is_dir() and (candidate / 'checkpoints').is_dir():
            return candidate
    raise FileNotFoundError('Could not find the sentiAvatar project root from ' + str(start))


PROJECT_ROOT = find_project_root(Path.cwd())
MOSHI_REPO = Path(os.environ.get('MOSHI_REPO', PROJECT_ROOT.parent / 'moshi')).expanduser().resolve()
MOSHI_PACKAGE_ROOT = MOSHI_REPO / 'moshi'
MIMI_WEIGHT = Path(os.environ.get(
    'MIMI_WEIGHT',
    PROJECT_ROOT / 'checkpoints' / 'mimi' / 'tokenizer-e351c8d8-checkpoint125.safetensors',
)).expanduser().resolve()
SUSU_ROOT = Path(os.environ.get(
    'SUSU_ROOT', PROJECT_ROOT / 'SuSuInterActs' / 'SuSuInterActs'
)).expanduser().resolve()
QWEN_CHECKPOINT = Path(os.environ.get(
    'QWEN_CHECKPOINT', PROJECT_ROOT / 'checkpoints' / 'llm'
)).expanduser().resolve()

REQUESTED_DEVICE = os.environ.get('MIMI_DEVICE', 'cuda')
NUM_CODEBOOKS = 8             # retain all levels in precomputed data
PLANNER_CODEBOOK = 0          # Phase 1 initially consumes semantic q0 only
MAX_AUDIO_SECONDS = 4.0       # enough for meaningful tests without wasting memory
STREAM_TEST_FRAMES = 16       # 1.28 seconds at 12.5 Hz
DATASET_AUDIT_CLIPS = 256     # deterministic, evenly spaced subset
VERIFY_SHA256 = True
RUN_DECODE_LISTENING_TEST = True
EXPECTED_MIMI_SHA256 = '09b782f0629851a271227fb9d36db65c041790365f11bbe5d3d59369cf863f50'

print('project:', PROJECT_ROOT)
print('moshi source:', MOSHI_PACKAGE_ROOT)
print('mimi weight:', MIMI_WEIGHT)
print('dataset:', SUSU_ROOT)
print('qwen checkpoint:', QWEN_CHECKPOINT)


## 2. Environment and reusable checks

Only `sentencepiece` is an extra import needed by the local Moshi loader in this notebook. If this cell reports it missing, install the local package in the active environment, for example:

```bash
python -m pip install -e /path/to/moshi/moshi
```


In [ ]:
if str(MOSHI_PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(MOSHI_PACKAGE_ROOT))

required_modules = ['torch', 'numpy', 'scipy', 'safetensors', 'huggingface_hub', 'sentencepiece', 'moshi']
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing_modules:
    raise RuntimeError(
        'Missing packages: ' + ', '.join(missing_modules) +
        f'\nInstall the local Moshi package with: python -m pip install -e {MOSHI_PACKAGE_ROOT}'
    )

import numpy as np
import torch
from scipy.io import wavfile
from scipy.signal import resample_poly
from safetensors import safe_open
from moshi.models import loaders
import moshi

RESULTS: list[dict[str, str | bool]] = []


def record(name: str, passed: bool, detail: object = '') -> bool:
    status = 'PASS' if passed else 'FAIL'
    detail_text = str(detail)
    RESULTS.append({'test': name, 'passed': bool(passed), 'detail': detail_text})
    print(f'[{status}] {name}' + (f': {detail_text}' if detail_text else ''))
    return bool(passed)


def require(name: str, condition: bool, detail: object = '') -> None:
    if not record(name, condition, detail):
        raise AssertionError(f'{name}: {detail}')


if REQUESTED_DEVICE.startswith('cuda') and not torch.cuda.is_available():
    DEVICE = torch.device('cpu')
    print('CUDA was requested but is unavailable; using CPU. The tests will be slower.')
else:
    DEVICE = torch.device(REQUESTED_DEVICE)

print({
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'torch': torch.__version__,
    'moshi': moshi.__version__,
    'device': str(DEVICE),
})
require('Moshi API version', moshi.__version__ == '0.2.13', f'found {moshi.__version__}; expected 0.2.13')


## 3. Checkpoint identity

This catches partial downloads and accidentally selecting the large Moshi language-model checkpoint instead of Mimi.


In [ ]:
require('Mimi checkpoint exists', MIMI_WEIGHT.is_file(), MIMI_WEIGHT)
size_bytes = MIMI_WEIGHT.stat().st_size
require('Mimi checkpoint size', size_bytes == 384_644_900, f'{size_bytes:,} bytes')

if VERIFY_SHA256:
    digest = hashlib.sha256()
    with MIMI_WEIGHT.open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    actual_sha256 = digest.hexdigest()
    require('Mimi SHA-256', actual_sha256 == EXPECTED_MIMI_SHA256, actual_sha256)
else:
    print('SHA-256 skipped by configuration.')


## 4. Real dataset audit and one aligned sample

The repository README says 16 kHz, but the files must be treated as ground truth. This audits an evenly spaced subset using WAV headers, verifies modality pairing, and loads the first split entry.


In [ ]:
split_path = SUSU_ROOT / 'split' / 'all_file_list.txt'
require('Dataset split exists', split_path.is_file(), split_path)
clip_names = [line.strip() for line in split_path.read_text(encoding='utf-8').splitlines() if line.strip()]
require('Dataset split is nonempty', len(clip_names) > 0, f'{len(clip_names):,} clips')

audit_count = min(DATASET_AUDIT_CLIPS, len(clip_names))
audit_indices = sorted({round(i * (len(clip_names) - 1) / max(1, audit_count - 1)) for i in range(audit_count)})
sample_rates, channels, sample_widths = Counter(), Counter(), Counter()
missing_pairs: list[str] = []
for index in audit_indices:
    name = clip_names[index]
    audio_path = SUSU_ROOT / 'wav_data' / f'{name}.wav'
    motion_path = SUSU_ROOT / 'motion_data' / f'{name}.npy'
    if not audio_path.is_file() or not motion_path.is_file():
        missing_pairs.append(name)
        continue
    with wave.open(str(audio_path), 'rb') as handle:
        sample_rates[handle.getframerate()] += 1
        channels[handle.getnchannels()] += 1
        sample_widths[handle.getsampwidth()] += 1

require('Sampled audio/motion pairs exist', not missing_pairs, f'missing={len(missing_pairs)} / sampled={len(audit_indices)}')
require('Sampled audio is mono', set(channels) == {1}, dict(channels))
require('Sampled audio is 16-bit PCM', set(sample_widths) == {2}, dict(sample_widths))
record('Sample-rate audit (informational)', True, dict(sample_rates))

SAMPLE_NAME = clip_names[0]
sample_audio_path = SUSU_ROOT / 'wav_data' / f'{SAMPLE_NAME}.wav'
sample_motion_path = SUSU_ROOT / 'motion_data' / f'{SAMPLE_NAME}.npy'
sample_text_path = SUSU_ROOT / 'text_data' / 'motion2text.json'
annotations = json.loads(sample_text_path.read_text(encoding='utf-8'))
motion = np.load(sample_motion_path, allow_pickle=True).item()
source_sr, source_audio = wavfile.read(sample_audio_path)

if np.issubdtype(source_audio.dtype, np.integer):
    scale = float(max(abs(np.iinfo(source_audio.dtype).min), np.iinfo(source_audio.dtype).max))
    source_audio = source_audio.astype(np.float32) / scale
else:
    source_audio = source_audio.astype(np.float32)
if source_audio.ndim == 2:
    source_audio = source_audio.mean(axis=1)

motion_frames = int(motion['body'].shape[0])
audio_duration = len(source_audio) / source_sr
motion_duration = motion_frames / 20.0
duration_error = abs(audio_duration - motion_duration)
print({
    'name': SAMPLE_NAME,
    'annotation': annotations[SAMPLE_NAME],
    'audio_sr': source_sr,
    'audio_seconds': audio_duration,
    'motion_frames_20hz': motion_frames,
    'motion_seconds': motion_duration,
    'motion_shapes': {key: value.shape for key, value in motion.items()},
})
require('Sample waveform is finite', bool(np.isfinite(source_audio).all()), f'peak={np.abs(source_audio).max():.4f}')
require('Sample audio-motion duration alignment', duration_error <= 1 / 20, f'error={duration_error:.6f}s')


## 5. Deterministic 16/24 kHz preprocessing

Mimi requires mono 24 kHz float audio. Existing 24 kHz clips take the identity path. A synthetic 16 kHz signal exercises the resampling branch so it is tested even when the local dataset is already 24 kHz.


In [ ]:
def resample_audio(audio: np.ndarray, source_rate: int, target_rate: int = 24_000) -> np.ndarray:
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim != 1:
        raise ValueError(f'Expected mono [samples], got {audio.shape}')
    if source_rate == target_rate:
        return np.ascontiguousarray(audio)
    factor = math.gcd(int(source_rate), int(target_rate))
    output = resample_poly(audio, target_rate // factor, source_rate // factor)
    expected = round(len(audio) * target_rate / source_rate)
    if len(output) > expected:
        output = output[:expected]
    elif len(output) < expected:
        output = np.pad(output, (0, expected - len(output)))
    return np.ascontiguousarray(output, dtype=np.float32)


audio_24k = resample_audio(source_audio, source_sr, 24_000)
expected_actual_length = round(len(source_audio) * 24_000 / source_sr)
require('Actual audio resample length', len(audio_24k) == expected_actual_length, len(audio_24k))
require('Actual audio resample is finite', bool(np.isfinite(audio_24k).all()), f'peak={np.abs(audio_24k).max():.4f}')

synthetic_t = np.arange(round(1.37 * 16_000), dtype=np.float32) / 16_000
synthetic_16k = (0.2 * np.sin(2 * np.pi * 440 * synthetic_t)).astype(np.float32)
synthetic_a = resample_audio(synthetic_16k, 16_000, 24_000)
synthetic_b = resample_audio(synthetic_16k, 16_000, 24_000)
require('16-to-24 kHz length', len(synthetic_a) == round(len(synthetic_16k) * 1.5), len(synthetic_a))
require('Resampling determinism', np.array_equal(synthetic_a, synthetic_b), 'exact equality')


## 6. Load Mimi and verify its contract

The initial planner will consume q0, but preprocessing should retain all eight Moshi codebooks so later ablations do not require re-tokenizing audio. Mimi explicitly implements one semantic quantizer followed by acoustic residual quantizers.


In [ ]:
load_start = time.perf_counter()
mimi = loaders.get_mimi(
    filename=MIMI_WEIGHT,
    device=DEVICE,
    num_codebooks=NUM_CODEBOOKS,
)
mimi.eval()
load_seconds = time.perf_counter() - load_start

model_info = {
    'load_seconds': round(load_seconds, 3),
    'parameters': sum(parameter.numel() for parameter in mimi.parameters()),
    'sample_rate': mimi.sample_rate,
    'frame_rate': mimi.frame_rate,
    'frame_size': mimi.frame_size,
    'active_codebooks': mimi.num_codebooks,
    'total_codebooks': mimi.total_codebooks,
    'cardinality': mimi.cardinality,
    'semantic_codebooks': mimi.quantizer.n_q_semantic,
}
print(model_info)
require('Mimi sample rate', mimi.sample_rate == 24_000, mimi.sample_rate)
require('Mimi frame rate', mimi.frame_rate == 12.5, mimi.frame_rate)
require('Mimi frame size', mimi.frame_size == 1_920, mimi.frame_size)
require('Mimi active codebooks', mimi.num_codebooks == 8, mimi.num_codebooks)
require('Mimi cardinality', mimi.cardinality == 2_048, mimi.cardinality)
require('Mimi q0 is the semantic codebook', mimi.quantizer.n_q_semantic == 1, mimi.quantizer.n_q_semantic)


## 7. Offline tokenization, range, determinism, and q0 statistics


In [ ]:
test_samples = min(len(audio_24k), round(MAX_AUDIO_SECONDS * mimi.sample_rate))
waveform = torch.from_numpy(audio_24k[:test_samples]).to(device=DEVICE, dtype=torch.float32)[None, None]

encode_start = time.perf_counter()
with torch.inference_mode():
    codes = mimi.encode(waveform)
if DEVICE.type == 'cuda':
    torch.cuda.synchronize(DEVICE)
encode_seconds = time.perf_counter() - encode_start

expected_frames = math.ceil(waveform.shape[-1] / mimi.frame_size)
q0 = codes[:, PLANNER_CODEBOOK, :]
print({
    'waveform_shape': tuple(waveform.shape),
    'codes_shape': tuple(codes.shape),
    'codes_dtype': str(codes.dtype),
    'code_min': int(codes.min()),
    'code_max': int(codes.max()),
    'encode_seconds': round(encode_seconds, 3),
    'audio_seconds': round(waveform.shape[-1] / mimi.sample_rate, 3),
})
require('Offline code shape', tuple(codes.shape) == (1, 8, expected_frames), tuple(codes.shape))
require('Offline code dtype', codes.dtype == torch.long, codes.dtype)
require('Offline code range', int(codes.min()) >= 0 and int(codes.max()) < 2_048, f'[{int(codes.min())}, {int(codes.max())}]')

with torch.inference_mode():
    repeated_codes = mimi.encode(waveform)
require('Offline token determinism', torch.equal(codes, repeated_codes), 'exact token equality')

q0_cpu = q0.detach().cpu().flatten()
counts = torch.bincount(q0_cpu, minlength=2_048).float()
probabilities = counts[counts > 0] / counts.sum()
q0_entropy_bits = float(-(probabilities * probabilities.log2()).sum())
record('q0 activity (informational)', True, f'unique={int((counts > 0).sum())}/{q0_cpu.numel()} frames, entropy={q0_entropy_bits:.3f} bits')


## 8. Causal streaming equivalence

Streaming Mimi accepts only positive multiples of 1,920 samples. This test feeds one 80 ms frame at a time, matching online inference, and compares the result with offline causal encoding of the same prefix. Exact q0 equality is the critical gate; exact equality across all eight levels is also required here.


In [ ]:
available_complete_frames = waveform.shape[-1] // mimi.frame_size
stream_frames = min(STREAM_TEST_FRAMES, available_complete_frames)
require('Enough audio for streaming test', stream_frames >= 2, f'{stream_frames} complete frames')
stream_waveform = waveform[..., :stream_frames * mimi.frame_size]

with torch.inference_mode():
    offline_prefix_codes = mimi.encode(stream_waveform)
    streamed_chunks = []
    stream_start = time.perf_counter()
    with mimi.streaming(batch_size=1):
        for frame_index in range(stream_frames):
            start = frame_index * mimi.frame_size
            frame = stream_waveform[..., start:start + mimi.frame_size]
            frame_codes = mimi.encode(frame)
            if frame_codes.shape[-1] != 1:
                raise AssertionError(f'Stream frame {frame_index} returned {frame_codes.shape}')
            streamed_chunks.append(frame_codes)
    streamed_codes = torch.cat(streamed_chunks, dim=-1)
if DEVICE.type == 'cuda':
    torch.cuda.synchronize(DEVICE)
stream_seconds = time.perf_counter() - stream_start

all_match = float((streamed_codes == offline_prefix_codes).float().mean())
q0_match = float((streamed_codes[:, 0] == offline_prefix_codes[:, 0]).float().mean())
print({
    'streamed_shape': tuple(streamed_codes.shape),
    'q0_match_rate': q0_match,
    'all_codebook_match_rate': all_match,
    'stream_seconds': round(stream_seconds, 3),
    'stream_audio_seconds': round(stream_frames / mimi.frame_rate, 3),
})
require('Streaming q0 equals offline q0', torch.equal(streamed_codes[:, 0], offline_prefix_codes[:, 0]), f'match={q0_match:.6f}')
require('Streaming all levels equal offline', torch.equal(streamed_codes, offline_prefix_codes), f'match={all_match:.6f}')


## 9. Direct future-leakage test

Two inputs share an identical prefix and have deliberately different futures. A causal tokenizer must emit identical tokens before the edit boundary.


In [ ]:
causal_test_frames = min(codes.shape[-1], max(4, stream_frames))
edit_frame = causal_test_frames // 2
causal_samples = causal_test_frames * mimi.frame_size
original = waveform[..., :causal_samples]
perturbed = original.clone()
edit_sample = edit_frame * mimi.frame_size
perturbed[..., edit_sample:] = -torch.flip(perturbed[..., edit_sample:], dims=[-1])
require('Future perturbation actually changes audio', not torch.equal(original[..., edit_sample:], perturbed[..., edit_sample:]), f'edit at {edit_frame / mimi.frame_rate:.3f}s')

with torch.inference_mode():
    original_codes = mimi.encode(original)
    perturbed_codes = mimi.encode(perturbed)

past_original = original_codes[..., :edit_frame]
past_perturbed = perturbed_codes[..., :edit_frame]
future_difference = float((original_codes[..., edit_frame:] != perturbed_codes[..., edit_frame:]).float().mean())
require('No future leakage in q0', torch.equal(past_original[:, 0], past_perturbed[:, 0]), f'prefix frames={edit_frame}')
require('No future leakage in all levels', torch.equal(past_original, past_perturbed), f'prefix frames={edit_frame}')
record('Future edit changes later tokens (sanity)', future_difference > 0, f'difference={future_difference:.3%}')


## 10. Optional listening test: q0 versus all eight levels

Mimi labels q0 as semantic by architecture, but listening provides a useful coarse-information sanity check. q0 audio is not expected to sound clean. Do not turn subjective intelligibility from one clip into a hard training gate.


In [ ]:
if RUN_DECODE_LISTENING_TEST:
    from IPython.display import Audio, display

    with torch.inference_mode():
        decoded_q0 = mimi.decode(codes[:, :1])
        decoded_q0_to_q7 = mimi.decode(codes)
    q0_audio = decoded_q0[0, 0].float().cpu().numpy()[:waveform.shape[-1]]
    full_audio = decoded_q0_to_q7[0, 0].float().cpu().numpy()[:waveform.shape[-1]]
    require('q0 decode is finite', bool(np.isfinite(q0_audio).all()), f'samples={len(q0_audio)}')
    require('8-level decode is finite', bool(np.isfinite(full_audio).all()), f'samples={len(full_audio)}')
    print('Original')
    display(Audio(waveform[0, 0].float().cpu().numpy(), rate=mimi.sample_rate))
    print('q0 only (semantic/coarse)')
    display(Audio(q0_audio, rate=mimi.sample_rate))
    print('q0-q7 (semantic + acoustic residuals)')
    display(Audio(full_audio, rate=mimi.sample_rate))
else:
    print('Listening test skipped by configuration.')


## 11. Audio/body timing contract for fixed `[gap_3]`

The causal body codec converts 20 Hz motion into 10 Hz tokens. `[gap_3]` means three unselected body-token positions between adjacent anchors, so the anchor distance is four positions. This is exactly five Mimi frames and does not require interpolation.


In [ ]:
RAW_MOTION_FPS = Fraction(20, 1)
BODY_TOKEN_FPS = Fraction(10, 1)
MIMI_FPS = Fraction(25, 2)
GAP_TOKEN = '[gap_3]'
GAP_VALUE = 3
ANCHOR_DISTANCE_BODY_FRAMES = GAP_VALUE + 1
ANCHOR_PERIOD = Fraction(ANCHOR_DISTANCE_BODY_FRAMES, BODY_TOKEN_FPS)
MIMI_FRAMES_PER_ANCHOR = ANCHOR_PERIOD * MIMI_FPS

require('Fixed-gap duration is 0.4 seconds', ANCHOR_PERIOD == Fraction(2, 5), float(ANCHOR_PERIOD))
require('Fixed-gap maps to an integer Mimi count', MIMI_FRAMES_PER_ANCHOR.denominator == 1, MIMI_FRAMES_PER_ANCHOR)
require('Five Mimi frames per anchor period', MIMI_FRAMES_PER_ANCHOR == 5, MIMI_FRAMES_PER_ANCHOR)

alignment_preview = []
for anchor_number in range(8):
    body_index = anchor_number * ANCHOR_DISTANCE_BODY_FRAMES
    seconds = Fraction(body_index, BODY_TOKEN_FPS)
    mimi_boundary = seconds * MIMI_FPS
    alignment_preview.append({
        'anchor_number': anchor_number,
        'body_token_index': body_index,
        'time_seconds': float(seconds),
        'mimi_boundary_index': int(mimi_boundary),
    })
print(alignment_preview)

expected_body_token_frames = motion_frames // 2
expected_mimi_frames = math.ceil(len(audio_24k) / mimi.frame_size)
record('Selected clip timeline (informational)', True, {
    'raw_motion_frames': motion_frames,
    'body_token_frames': expected_body_token_frames,
    'mimi_frames': expected_mimi_frames,
    'audio_seconds': audio_duration,
})


## 12. Existing Qwen checkpoint and token-namespace preflight

This is read-only and does not load the full language model. It verifies tokenizer/config/embedding row consistency and inspects the legacy token namespaces. The planned multipart mapping is `slot × 512 + code`, giving 16 × 512 = 8,192 body IDs. Existing audio text tokens are **not** the planned Mimi input; Phase 1 will use a separate 2,048 × hidden-size Mimi embedding.


In [ ]:
from transformers import AutoTokenizer

qwen_config_path = QWEN_CHECKPOINT / 'config.json'
qwen_weight_path = QWEN_CHECKPOINT / 'model.safetensors'
qwen_added_path = QWEN_CHECKPOINT / 'added_tokens.json'
require('Qwen config exists', qwen_config_path.is_file(), qwen_config_path)
require('Qwen weights exist', qwen_weight_path.is_file(), qwen_weight_path)
require('Qwen added-token map exists', qwen_added_path.is_file(), qwen_added_path)

qwen_config = json.loads(qwen_config_path.read_text(encoding='utf-8'))
qwen_added = json.loads(qwen_added_path.read_text(encoding='utf-8'))
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_CHECKPOINT, local_files_only=True)
with safe_open(qwen_weight_path, framework='pt', device='cpu') as handle:
    embedding_shape = tuple(handle.get_slice('model.embed_tokens.weight').get_shape())

require('Qwen tokenizer/config vocab agreement', len(qwen_tokenizer) == qwen_config['vocab_size'], f"tokenizer={len(qwen_tokenizer)}, config={qwen_config['vocab_size']}")
require('Qwen embedding/config vocab agreement', embedding_shape[0] == qwen_config['vocab_size'], f'embedding={embedding_shape}, config={qwen_config["vocab_size"]}')
require('Qwen hidden size is 896', embedding_shape[1] == 896 and qwen_config['hidden_size'] == 896, embedding_shape)

def numbered_namespace(prefix: str) -> list[tuple[int, int]]:
    values = []
    for token, token_id in qwen_added.items():
        if token.startswith(prefix) and token.endswith(']'):
            try:
                values.append((int(token[len(prefix):-1]), int(token_id)))
            except ValueError:
                pass
    return sorted(values)

body_namespace = numbered_namespace('[body_')
audio_namespace = numbered_namespace('[audio_')
planned_body_tokens = [f'[body_{index}]' for index in range(16 * 512)]
missing_planned_body = [token for token in planned_body_tokens if token not in qwen_added]
planned_body_ids = [qwen_added[token] for token in planned_body_tokens if token in qwen_added]
require('All 8,192 multipart body IDs already exist', not missing_planned_body, f'missing={len(missing_planned_body)}')
require('Planned multipart body IDs are unique', len(set(planned_body_ids)) == 8_192, f'unique={len(set(planned_body_ids))}')
record('Legacy token namespaces (informational)', True, {
    'body_tokens': len(body_namespace),
    'audio_tokens': len(audio_namespace),
    'body_logical_range': (body_namespace[0][0], body_namespace[-1][0]),
    'audio_logical_range': (audio_namespace[0][0], audio_namespace[-1][0]),
    'planned_mimi_embedding_shape': (2_048, qwen_config['hidden_size']),
})

existing_gap_tokens = sorted(token for token in qwen_added if token.startswith('[gap_'))
record('Gap/control tokens pending Phase 1 implementation', True, existing_gap_tokens or 'none currently present')
print('Important: existing [audio_*] tokenizer symbols are legacy tokens; do not use them as Mimi q0 inputs.')


## 13. Final go/no-go report

Phase 1 audio implementation can begin only when every hard test below passes. Informational rows describe the observed data rather than impose a threshold.


In [ ]:
try:
    import pandas as pd
    from IPython.display import display
    report = pd.DataFrame(RESULTS)
    display(report)
except ImportError:
    report = RESULTS
    for row in report:
        print(row)

failed = [row for row in RESULTS if not row['passed']]
print()
if failed:
    print(f'NO-GO: {len(failed)} test(s) failed. Fix them before implementing Phase 1.')
else:
    print('GO: all Mimi, causality, timing, dataset, and Qwen preflight tests passed.')
print('Reminder: this validates the representation pipeline, not whether q0 predicts motion well. That requires the Phase 1 baseline and q0-vs-multi-level ablation.')
